# Clasificación de animales con CNN y Transfer Learning

En este notebook se clasifican imágenes del dataset de animales de Kaggle (15 clases: Bear, Bird, Cat, Cow, Deer, Dog, Dolphin, Elephant, Giraffe, Horse, Kangaroo, Lion, Panda, Tiger y Zebra).

Se siguen los mismos pasos de los notebooks de dígitos que ya trae NNApp (cargar datos, dividir, crear el modelo, entrenar, evaluar y probar con una imagen), pero ahora con imágenes a color y en los dos frameworks de la app:

| Parte | Qué se hace |
|---|---|
| A. TensorFlow | CNN propia con imágenes de 32x32 y transfer learning con ResNet50V2 |
| B. PyTorch | CNN propia con imágenes de 32x32 y transfer learning con ResNet18 |
| C. Comparación | Tabla con accuracy, parámetros y tiempo, curvas de aprendizaje y matriz de confusión |
| D. Guardar y probar | Se guardan los pesos de los 4 modelos y se cargan de nuevo para clasificar imágenes nuevas |

**Cómo correrlo**
- **En local:** deja el notebook en la carpeta `notebooks/` del proyecto NNApp y ábrelo desde ahí. Los pesos se guardan solos en `models/saved/`, que es donde los busca la app.
- **En Colab:** sube `archive.zip` (el dataset de Kaggle) al panel de archivos y ejecuta todo. Al final se crea un `models_saved.zip` para descargarlo y copiar su contenido en `models/saved/` de NNApp.

## 0. Configuración

In [ ]:
import os, time, json, copy, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)

# Si el notebook está en NNApp/notebooks usamos la carpeta del proyecto (..),
# si no (por ejemplo en Colab) usamos la carpeta actual.
ROOT = Path("..") if Path("../models").exists() else Path(".")
DATA_DIR = ROOT / "datasets" / "animal_data"
SAVE_DIR = ROOT / "models" / "saved"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CNN_SIZE = 32        # tamaño pedido en el taller para la CNN
TRANSFER_SIZE = 128  # ResNet viene entrenada con imágenes grandes, con 128 rinde mucho mejor que con 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("TensorFlow:", tf.__version__, "| PyTorch:", torch.__version__, "| dispositivo PyTorch:", device)
print("Datos en:", DATA_DIR.resolve(), "| pesos en:", SAVE_DIR.resolve())

## 1. Datos

El dataset trae una carpeta por animal. Si todavía no está descomprimido, se saca de `archive.zip`.

In [ ]:
if not DATA_DIR.exists():
    zip_path = ROOT / "archive.zip"
    if zip_path.exists():
        import zipfile
        (ROOT / "datasets").mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(ROOT / "datasets")
    else:
        raise FileNotFoundError(
            "No encuentro el dataset. Descarga https://www.kaggle.com/datasets/likhon148/animal-data "
            "y deja archive.zip junto al notebook (o la carpeta datasets/animal_data en el proyecto)."
        )

class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
num_classes = len(class_names)

paths, labels = [], []
for idx, name in enumerate(class_names):
    for f in sorted((DATA_DIR / name).iterdir()):
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            paths.append(f)
            labels.append(idx)
labels = np.array(labels)

print(f"{len(paths)} imágenes en {num_classes} clases")
print(pd.Series(labels).map(dict(enumerate(class_names))).value_counts().sort_index().to_string())

La función `load_image` hace lo mismo que `preprocess_image` de la app (RGB, `resize` y pasar a [0, 1]). Así, una imagen se prepara igual al entrenar y al predecir.

In [ ]:
def load_image(path, size):
    img = Image.open(path).convert("RGB").resize((size, size))
    return np.array(img).astype("float32") / 255.0

X32 = np.stack([load_image(p, CNN_SIZE) for p in paths])          # para la CNN (32x32)
XT  = np.stack([load_image(p, TRANSFER_SIZE) for p in paths])     # para ResNet
print("X32:", X32.shape, "| XT:", XT.shape)

# Un ejemplo de cada clase, a 32x32 (lo que ve la CNN) 
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for ax, c in zip(axes.ravel(), range(num_classes)):
    i = np.where(labels == c)[0][0]
    ax.imshow(X32[i]); ax.set_title(class_names[c]); ax.axis("off")
plt.tight_layout(); plt.show()

Se divide en train (70 %), validación (15 %) y test (15 %), igual que en el notebook de dígitos. La división se hace una sola vez sobre los índices, así los cuatro modelos usan exactamente las mismas imágenes de test.

In [ ]:
idx_all = np.arange(len(labels))
idx_train, idx_tmp = train_test_split(idx_all, test_size=0.3, random_state=SEED, stratify=labels)
idx_val, idx_test  = train_test_split(idx_tmp, test_size=0.5, random_state=SEED, stratify=labels[idx_tmp])

y_train, y_val, y_test = labels[idx_train], labels[idx_val], labels[idx_test]

X32_train, X32_val, X32_test = X32[idx_train], X32[idx_val], X32[idx_test]
XT_train,  XT_val,  XT_test  = XT[idx_train],  XT[idx_val],  XT[idx_test]
print(f"Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}")

# Diccionario donde se van guardando los resultados de cada modelo para la comparación
results = {}

def save_results(name, framework, size, probs, y_true, train_time, n_params, history):
    pred = probs.argmax(axis=1)
    rep = classification_report(y_true, pred, target_names=class_names, output_dict=True, zero_division=0)
    results[name] = dict(framework=framework, size=size, acc=rep["accuracy"], f1=rep["macro avg"]["f1-score"],
                         params=n_params, time=train_time, history=history, pred=pred)
    print(f"{name}: accuracy test = {rep['accuracy']:.4f} | F1 macro = {rep['macro avg']['f1-score']:.4f} "
          f"| parámetros = {n_params:,} | tiempo = {train_time:.0f} s")

## A. TensorFlow

### A.1 CNN propia (32x32)

Es la CNN de `tensorflow_arch.py` (Conv, MaxPooling, Flatten, Dense) con una convolución más, `Dropout` y un giro horizontal aleatorio. El giro y el dropout ayudan porque hay pocas imágenes (unas 1360 para entrenar). La salida es `softmax` con 15 clases.

In [ ]:
def build_tf_cnn(num_classes, size=32):
    model = keras.Sequential([
        layers.Input(shape=(size, size, 3)),
        layers.RandomFlip("horizontal"),            # solo actúa al entrenar
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

tf_cnn = build_tf_cnn(num_classes, CNN_SIZE)
tf_cnn.summary()

In [ ]:
t0 = time.time()
h = tf_cnn.fit(
    X32_train, y_train,
    validation_data=(X32_val, y_val),
    epochs=60, batch_size=32, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)],
)
tf_cnn_time = time.time() - t0

probs = tf_cnn.predict(X32_test, verbose=0)
save_results("CNN (TensorFlow)", "tensorflow", CNN_SIZE, probs, y_test, tf_cnn_time, tf_cnn.count_params(), h.history)

### A.2 Transfer learning con ResNet50V2

Se usa ResNet50V2 ya entrenada con ImageNet, sin su última capa, y se le pone encima una capa nueva para las 15 clases de animales. Se entrena en dos pasos:
1. Con la ResNet congelada, solo se entrena la capa nueva.
2. Se descongela toda la red y se entrena un poco más con un learning rate muy bajo (fine-tuning).

La entrada de la app va en [0, 1], así que dentro del modelo hay una capa `Rescaling` que la pasa a [-1, 1], que es lo que espera ResNet50V2.

In [ ]:
def build_tf_transfer(num_classes, size=128):
    base = keras.applications.ResNet50V2(include_top=False, weights="imagenet", input_shape=(size, size, 3))
    base.trainable = False

    inputs = keras.Input(shape=(size, size, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.Rescaling(2.0, offset=-1.0)(x)     # [0,1] -> [-1,1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs), base

def compile_tf(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

tf_resnet, tf_base = build_tf_transfer(num_classes, TRANSFER_SIZE)
compile_tf(tf_resnet, 1e-3)

early = lambda: keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)

t0 = time.time()
# Paso 1: solo la capa nueva
h1 = tf_resnet.fit(XT_train, y_train, validation_data=(XT_val, y_val),
                   epochs=10, batch_size=32, verbose=2, callbacks=[early()])

# Paso 2: fine-tuning de toda la red
tf_base.trainable = True
compile_tf(tf_resnet, 1e-5)
h2 = tf_resnet.fit(XT_train, y_train, validation_data=(XT_val, y_val),
                   epochs=10, batch_size=32, verbose=2, callbacks=[early()])
tf_resnet_time = time.time() - t0

hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
probs = tf_resnet.predict(XT_test, verbose=0)
save_results("ResNet50V2 (TensorFlow)", "tensorflow", TRANSFER_SIZE, probs, y_test, tf_resnet_time, tf_resnet.count_params(), hist)

### A.3 Guardar los pesos (TensorFlow)

Se guarda el modelo completo (`.keras`, es el que carga la app) y también solo los pesos (`.weights.h5`).

In [ ]:
tf_cnn.save(str(SAVE_DIR / "tf_image_cnn.keras"))
tf_cnn.save_weights(str(SAVE_DIR / "tf_image_cnn.weights.h5"))
tf_resnet.save(str(SAVE_DIR / "tf_image_transfer.keras"))
tf_resnet.save_weights(str(SAVE_DIR / "tf_image_transfer.weights.h5"))
print(sorted(p.name for p in SAVE_DIR.glob("tf_image_*")))

## B. PyTorch

Mismos modelos y mismo entrenamiento, pero en PyTorch. Las clases `AnimalCNN` y `AnimalResNet` están copiadas igual en `models/pytorch_arch.py`, porque la app las necesita para cargar los pesos.

In [ ]:
class AnimalCNN(nn.Module):
    # Imágenes de 32x32 -> tres MaxPool dejan 4x4
    def __init__(self, num_classes=15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.net(x)


class AnimalResNet(nn.Module):
    # ResNet18 con la última capa cambiada. Recibe imágenes en [0,1] y las normaliza por dentro (ImageNet).
    def __init__(self, num_classes=15, pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.backbone(x)


def to_torch(X):
    # (N, H, W, 3) -> (N, 3, H, W)
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous()

T32_train, T32_val, T32_test = to_torch(X32_train), to_torch(X32_val), to_torch(X32_test)
TT_train,  TT_val,  TT_test  = to_torch(XT_train),  to_torch(XT_val),  to_torch(XT_test)
ty_train, ty_val, ty_test = [torch.tensor(y, dtype=torch.long) for y in (y_train, y_val, y_test)]

Funciones para entrenar y evaluar. Guardan el mejor modelo según el accuracy de validación (igual que `restore_best_weights` en Keras).

In [ ]:
def predict_torch(model, X, batch=128):
    # devuelve los logits de todas las imágenes
    model.to(device).eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            outs.append(model(X[i:i + batch].to(device)).cpu())
    return torch.cat(outs)


def fit_torch(model, X_tr, y_tr, X_va, y_va, epochs, lr, patience=10):
    model.to(device)
    opt = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_acc, best_state, wait = -1.0, None, 0

    for ep in range(epochs):
        model.train()
        tot_loss, correct = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            flip = torch.rand(len(xb), device=device) < 0.5          # giro horizontal aleatorio
            xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            tot_loss += loss.item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()

        logits = predict_torch(model, X_va)
        val_loss = loss_fn(logits, y_va).item()
        val_acc = (logits.argmax(1) == y_va).float().mean().item()
        hist["loss"].append(tot_loss / len(X_tr));  hist["accuracy"].append(correct / len(X_tr))
        hist["val_loss"].append(val_loss);          hist["val_accuracy"].append(val_acc)
        print(f"Epoch {ep + 1}/{epochs} - loss: {hist['loss'][-1]:.4f} - accuracy: {hist['accuracy'][-1]:.4f} "
              f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, best_state, wait = val_acc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping")
                break

    model.load_state_dict(best_state)
    return hist


def count_params(model):
    return sum(p.numel() for p in model.parameters())

### B.1 CNN propia (32x32)

In [ ]:
pt_cnn = AnimalCNN(num_classes)

t0 = time.time()
h = fit_torch(pt_cnn, T32_train, ty_train, T32_val, ty_val, epochs=60, lr=1e-3, patience=10)
pt_cnn_time = time.time() - t0

probs = torch.softmax(predict_torch(pt_cnn, T32_test), dim=1).numpy()
save_results("CNN (PyTorch)", "pytorch", CNN_SIZE, probs, y_test, pt_cnn_time, count_params(pt_cnn), h)

### B.2 Transfer learning con ResNet18

Igual que en TensorFlow: primero se entrena solo la capa `fc` nueva con la red congelada y después se descongela todo con un learning rate más bajo.

In [ ]:
pt_resnet = AnimalResNet(num_classes, pretrained=True)

t0 = time.time()
# Paso 1: solo la capa nueva
for p in pt_resnet.parameters():
    p.requires_grad = False
for p in pt_resnet.backbone.fc.parameters():
    p.requires_grad = True
h1 = fit_torch(pt_resnet, TT_train, ty_train, TT_val, ty_val, epochs=10, lr=1e-3, patience=4)

# Paso 2: fine-tuning de toda la red
for p in pt_resnet.parameters():
    p.requires_grad = True
h2 = fit_torch(pt_resnet, TT_train, ty_train, TT_val, ty_val, epochs=10, lr=1e-4, patience=4)
pt_resnet_time = time.time() - t0

hist = {k: h1[k] + h2[k] for k in h1}
probs = torch.softmax(predict_torch(pt_resnet, TT_test), dim=1).numpy()
save_results("ResNet18 (PyTorch)", "pytorch", TRANSFER_SIZE, probs, y_test, pt_resnet_time, count_params(pt_resnet), hist)

### B.3 Guardar los pesos (PyTorch)

Se guarda el `state_dict` de cada modelo, que es lo que carga `models/pytorch_models.py`.

In [ ]:
def save_state(model, name):
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, SAVE_DIR / name)

save_state(pt_cnn, "pt_image_cnn.pt")
save_state(pt_resnet, "pt_image_transfer.pt")
print(sorted(p.name for p in SAVE_DIR.glob("pt_image_*")))

## C. Comparación de los modelos

Todos se evalúan con las mismas imágenes de test. La CNN usa imágenes de 32x32 y las ResNet de 128x128 (ver la configuración del inicio).

In [ ]:
comparison = pd.DataFrame([
    {"Modelo": n, "Framework": r["framework"], "Tamaño imagen": f"{r['size']}x{r['size']}",
     "Parámetros": r["params"], "Tiempo de entrenamiento (s)": round(r["time"]),
     "Accuracy test": round(r["acc"], 4), "F1 macro": round(r["f1"], 4)}
    for n, r in results.items()
]).sort_values("Accuracy test", ascending=False).reset_index(drop=True)
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) accuracy en test
axes[0].barh(comparison["Modelo"][::-1], comparison["Accuracy test"][::-1], color="#4c78a8")
axes[0].set_xlim(0, 1); axes[0].set_title("Accuracy en test")
for i, v in enumerate(comparison["Accuracy test"][::-1]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center")

# 2) y 3) curvas de validación
for n, r in results.items():
    axes[1].plot(r["history"]["val_accuracy"], label=n)
    axes[2].plot(r["history"]["val_loss"], label=n)
axes[1].set_title("Accuracy de validación por época"); axes[1].set_xlabel("época"); axes[1].legend(fontsize=8)
axes[2].set_title("Loss de validación por época");     axes[2].set_xlabel("época")
plt.tight_layout(); plt.show()

In [ ]:
best = comparison.loc[0, "Modelo"]
print("Mejor modelo en test:", best)
print(classification_report(y_test, results[best]["pred"], target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix(y_test, results[best]["pred"]), display_labels=class_names).plot(
    ax=ax, xticks_rotation=60, colorbar=False)
ax.set_title(f"Matriz de confusión - {best}")
plt.tight_layout(); plt.show()

**Cómo leer la comparación**

- La CNN propia empieza de cero con solo unas 1360 imágenes de 32x32. Por eso suele quedarse en un accuracy medio y se nota mucho la diferencia entre accuracy de train y de validación (overfitting).
- Las ResNet ya vienen entrenadas con ImageNet, que tiene muchos animales. Solo hay que adaptarlas a las 15 clases, por eso normalmente aprenden más rápido y llegan a un accuracy bastante más alto.
- A cambio, las ResNet tienen muchos más parámetros y tardan más en entrenar.
- Entre TensorFlow y PyTorch las diferencias suelen ser pequeñas, porque los modelos y el entrenamiento son equivalentes.

*(Después de correr el notebook, cambia este texto por lo que realmente salió en la tabla de arriba.)*

## D. Guardar la información para la app y probar con imágenes nuevas

La app necesita saber los nombres de las clases y con qué tamaño se entrenó cada modelo. Eso se guarda en `animals_meta.json`.

In [ ]:
meta = {"classes": class_names, "cnn_size": CNN_SIZE, "transfer_size": TRANSFER_SIZE}
with open(SAVE_DIR / "animals_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
comparison.to_csv(SAVE_DIR / "comparacion_modelos.csv", index=False)

print(sorted(p.name for p in SAVE_DIR.iterdir()))

Ahora se cargan los pesos guardados en modelos nuevos (como lo hace la app) y se usan para clasificar imágenes que no se usaron al entrenar. Para probar con tus propias fotos, cambia la lista `fotos` por las rutas de tus imágenes.

In [ ]:
# --- cargar los 4 modelos desde los archivos guardados ---
tf_cnn_loaded    = keras.models.load_model(str(SAVE_DIR / "tf_image_cnn.keras"))
tf_resnet_loaded = keras.models.load_model(str(SAVE_DIR / "tf_image_transfer.keras"))

pt_cnn_loaded = AnimalCNN(num_classes)
pt_cnn_loaded.load_state_dict(torch.load(SAVE_DIR / "pt_image_cnn.pt", map_location="cpu"))
pt_resnet_loaded = AnimalResNet(num_classes, pretrained=False)     # sin descargar nada, solo la arquitectura
pt_resnet_loaded.load_state_dict(torch.load(SAVE_DIR / "pt_image_transfer.pt", map_location="cpu"))


def classify_image(path):
    # devuelve la clase que predice cada modelo para una imagen cualquiera
    x32, xt = load_image(path, CNN_SIZE), load_image(path, TRANSFER_SIZE)
    probs = {
        "CNN (TF)":      tf_cnn_loaded.predict(x32[None], verbose=0)[0],
        "ResNet (TF)":   tf_resnet_loaded.predict(xt[None], verbose=0)[0],
        "CNN (PT)":      torch.softmax(predict_torch(pt_cnn_loaded, to_torch(x32[None])), dim=1)[0].numpy(),
        "ResNet (PT)":   torch.softmax(predict_torch(pt_resnet_loaded, to_torch(xt[None])), dim=1)[0].numpy(),
    }
    return {m: (class_names[p.argmax()], float(p.max())) for m, p in probs.items()}


# 4 imágenes del conjunto de test; cambia esta lista por tus propias fotos si quieres
fotos = [paths[i] for i in idx_test[:4]]

fig, axes = plt.subplots(1, len(fotos), figsize=(4.5 * len(fotos), 5))
for ax, foto in zip(np.atleast_1d(axes), fotos):
    pred = classify_image(foto)
    ax.imshow(Image.open(foto).convert("RGB")); ax.axis("off")
    ax.set_title(f"Real: {foto.parent.name}\n" + "\n".join(f"{m}: {c} ({p:.0%})" for m, (c, p) in pred.items()), fontsize=9)
plt.tight_layout(); plt.show()

Por último, un zip con los pesos para llevarlos a la app (útil si se entrenó en Colab).

In [ ]:
import shutil
shutil.make_archive("models_saved", "zip", SAVE_DIR)
print("Creado models_saved.zip -> descomprímelo dentro de NNApp/models/saved/")